In [1]:
import numpy as np
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from src.models.sequential import Sequential
from src.layers import Conv1D, Dense
from src.layers.activation import Flatten, ReLU, Softmax
from src.loss.cross_entropy import CrossEntropy
from src.optimizer.adam import Adam
from src.backend import backend, BackendPolicy
import time
from src.models.neural_network import Network


In [ ]:
def prepare_har_dataset(csv_path: str) -> tuple[np.ndarray, np.ndarray, np.ndarray, int]:
    raw_data = np.genfromtxt(csv_path, delimiter=",")
    Y = raw_data[:, 9000].astype("int32")
    features_raw = raw_data[:, :9000].astype("float32")
    lengths = raw_data[:, 9001]

    num_samples = features_raw.shape[0]
    num_channels = 6
    time_steps = 1500
    X = np.zeros((num_samples, time_steps, num_channels), dtype="float32")
    for i in range(num_channels):
        X[:, :, i] = features_raw[:, i*1500 : (i+1)*1500]
    mask = np.arange(time_steps) < lengths[:, None]
    active_data = X[mask]
    scaler = StandardScaler()
    scaler.fit(active_data)
    X_flat = X.reshape(-1, num_channels)
    X_scaled_flat = scaler.transform(X_flat)
    X_final = X_scaled_flat.reshape(num_samples, time_steps, num_channels)
    X_final[~mask] = 0.0
    return X_final, Y, lengths, X_final.nbytes + Y.nbytes



In [3]:
dataset_path = r"data\3.Time_domain_subsamples\KU-HAR_v1.0_raw_samples.csv"
num_classes = 18
activity_map = {
    0: "Stand", 1: "Sit", 2: "Talk-sit", 3: "Talk-stand", 4: "Stand-sit",
    5: "Lay", 6: "Lay-stand", 7: "Pick", 8: "Jump", 9: "Push-up",
    10: "Sit-up", 11: "Walk", 12: "Walk-backward", 13: "Walk-circle",
    14: "Run", 15: "Stair-up", 16: "Stair-down", 17: "Table-tennis"
}


In [ ]:

X,Y, lengths, total_bytes = prepare_har_dataset(dataset_path)
Y = np.eye(num_classes)[Y]

In [5]:
model = Sequential([
    Conv1D(filters=16, kernel_size=7, stride=2, padding=3, name="conv1"),
    ReLU(),
    Conv1D(filters=32, kernel_size=5, stride=2, padding=2, name="conv2"),
    ReLU(),
    Flatten(),
    Dense(18),
    Softmax()
])

model.compile(optimizer=Adam(learning_rate=0.001), loss=CrossEntropy())

In [10]:

def compare_gpu_vs_cpu(X, Y, model: Network, epochs=1, batch_size=64, verbose=5):
    if not backend._cupy_runtime_ok:
        print("GPU runtime not available. Skipping GPU vs CPU comparison.")
        return -1,-1
    backend.configure(policy=BackendPolicy(use_gpu=False), dataset_bytes=0)
    model.reset()
    X_cpu, Y_cpu = backend.asarray(X), backend.asarray(Y)
    _, cpu_time = model.fit(X_cpu, Y_cpu, epochs=epochs, batch_size=batch_size, verbose=verbose)
    model.reset()
    backend.configure(policy=BackendPolicy(use_gpu=True, min_gpu_bytes=0), dataset_bytes=10)
    X_gpu, Y_gpu = backend.asarray(X), backend.asarray(Y)
    _, gpu_time = model.fit(X_gpu, Y_gpu, epochs=epochs, batch_size=batch_size, verbose=verbose)
    return cpu_time, gpu_time

cpu_time, gpu_time = compare_gpu_vs_cpu(X, Y, model, epochs=25, batch_size=1024, verbose=5)
print(f"CPU time: {cpu_time:.2f} seconds")
print(f"GPU time: {gpu_time:.2f} seconds")
print(f"Speedup: {cpu_time / gpu_time:.2f}x")



2026-03-21 12:37:25,635 | INFO | src.backend | Using backend: numpy
Epoch 1/25 - Loss: 2.906605 - 4.72s/epoch - ETA: 00:01:53
Epoch 6/25 - Loss: 1.632798 - 4.62s/epoch - ETA: 00:01:29
Epoch 11/25 - Loss: 0.976136 - 4.66s/epoch - ETA: 00:01:05
Epoch 16/25 - Loss: 0.715782 - 4.63s/epoch - ETA: 00:00:42
Epoch 21/25 - Loss: 0.578900 - 4.64s/epoch - ETA: 00:00:18
Epoch 25/25 - Loss: 0.507553 - 4.64s/epoch - ETA: 00:00:00

Training Complete. Total Time: 00:01:57
2026-03-21 12:39:22,891 | INFO | src.backend | GPU requested. CuPy available: True.
2026-03-21 12:39:22,891 | INFO | src.backend | Using backend: cupy
Epoch 1/25 - Loss: 2.923100 - 0.71s/epoch - ETA: 00:00:16
Epoch 6/25 - Loss: 1.571183 - 0.34s/epoch - ETA: 00:00:07
Epoch 11/25 - Loss: 0.986485 - 0.34s/epoch - ETA: 00:00:05
Epoch 16/25 - Loss: 0.736586 - 0.34s/epoch - ETA: 00:00:03
Epoch 21/25 - Loss: 0.601169 - 0.34s/epoch - ETA: 00:00:01
Epoch 25/25 - Loss: 0.527595 - 0.34s/epoch - ETA: 00:00:00

Training Complete. Total Time: 00:0